# 05 · 메모리 관리 & 성능측정

> **CuPy 2일 집중 코스 — Day 1 / 단원 3 (메모리 관리 및 성능측정, 1.5H)**

GPU 성능의 절반은 **데이터 이동**과 **메모리 사용**에서 결정됩니다. 암묵적 전송을 피하고,
메모리풀과 임시버퍼를 다루며, 프로파일링으로 병목을 찾는 법을 배웁니다.

### 왜 메모리 관리가 곧 성능인가

`00_intro_env`에서 GPU를 "수천 인승 고속버스 함대"에 비유했습니다. 문제는 이 버스 함대가 아무리
빨리 달려도 **승객(데이터)을 태우고 내리는 시간**(전송)과 **버스를 대기시키는 시간**(할당)이
크면 총 이동 시간은 여전히 느리다는 점입니다. 실전 CuPy 코드에서 성능 저하의 원인은 커널 연산
자체보다 다음 세 가지에서 오는 경우가 압도적으로 많습니다.
1. **불필요한 host↔device 전송** — PCIe 대역폭은 GPU 내부 메모리 대역폭의 1/10~1/50 수준.
2. **잦은 `cudaMalloc`/`cudaFree`** — 디바이스 동기화를 유발하는 무거운 시스템 콜.
3. **과도한 임시 배열(temporary)** — 커널 하나당 HBM read/write가 늘어나 메모리 대역폭을 갉아먹음.

이 노트북은 이 세 가지를 각각 진단하고 고치는 도구(메모리풀 관찰, `out=`, 프로파일러)를 다룹니다.

### 이 노트북의 흐름

host/device 메모리 공간 복습 → 암묵적 전송 함정 재확인 → **메모리풀의 내부 동작** → `out=`으로
임시버퍼 줄이기 → 전송 병목을 더 큰 데이터로 재확인 → `benchmark`/`time_range`/`profile` 3종
프로파일링 도구 → CUB 가속 백엔드 → 연습문제(파이프라인 2배 개선) → 실전 예제(Power Iteration).
`00`에서 다룬 "비동기 타이밍의 함정"과 "전송 비용"을 이 노트북에서 메모리 관점으로 확장한다고
생각하면 됩니다.

## 학습 목표
- host/device 메모리 공간과 전송(`asarray`/`asnumpy`)을 이해하고 **데이터를 GPU에서 생성**한다.
- **암묵적 전송·동기화**를 유발하는 연산을 식별하고 피한다.
- **메모리풀**의 동작 원리(caching allocator)를 이해하고, `used_bytes`/`total_bytes`를 관찰한다.
- `CUPY_GPU_MEMORY_LIMIT`과 커스텀 할당자로 메모리 사용을 제어하는 방법을 안다.
- `out=`으로 임시버퍼(temporary)를 줄여 메모리 대역폭을 절약한다.
- `benchmark`/`time_range`/`profile` 로 병목을 측정·표시하고, CUB 가속(`CUPY_ACCELERATORS`)을 활용한다.


## 목차
1. [host/device 메모리 공간](#1)
2. [암묵적 전송 주의](#2)
3. [메모리풀 관찰](#3)
4. [임시버퍼 줄이기 (out=)](#4)
5. [전송 병목 비교](#5)
6. [프로파일링 도구](#6)
7. [연습문제](#7)
8. [(실전) Power Iteration](#8)
9. [체크포인트](#9)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare, bytes_human
print_env()

<a id="1"></a>
## 1. host/device 메모리 공간

이기종(heterogeneous) 시스템은 **두 메모리 공간**으로 나뉩니다: CPU가 접근하는 **Host Memory**, GPU가 접근하는 **Device Memory**.
연산하려면 데이터가 해당 프로세서의 메모리에 있어야 하므로 **명시적 전송**이 필요합니다.
- Host → Device: `x_dev = cp.asarray(x_host)`
- Device → Host: `y_host = cp.asnumpy(y_dev)`

### 조금 더 구체적으로

`cp.asarray`/`cp.asnumpy`는 내부적으로 **`cudaMemcpy`**(또는 비동기 버전 `cudaMemcpyAsync`)를
호출해 PCIe(또는 NVLink) 버스로 바이트를 복사합니다. 기본적으로 이 복사는 **기본 스트림(default
stream)** 위에서 실행되며 CPU 쪽 배열이 **pageable memory**(일반 `malloc`으로 잡힌, OS가 스와핑할
수 있는 메모리)라면 CUDA 드라이버가 내부적으로 임시 **pinned(고정) 버퍼**를 거쳐 2단계로 복사합니다
— 이 때문에 매번 추가 지연이 생깁니다. `cudaHostAlloc`으로 미리 확보한 **pinned memory**를 쓰면
이 중간 단계가 사라져 전송이 더 빠르고, `cudaMemcpyAsync`와 결합하면 **연산과 전송을 겹칠(overlap)**
수도 있습니다 — 이 최적화는 `06_streams_async`에서 스트림·이벤트와 함께 본격적으로 다룹니다.

> 지금 단계에서 기억할 것: **전송은 공짜가 아니고, 방향(H2D/D2H)에 상관없이 대칭적으로 비쌉니다.**
> `00_intro_env` 7절에서 측정했듯 PCIe Gen4 x16 기준 이론 대역폭은 편도 약 32GB/s로, GPU 내부
> 대역폭(수백 GB/s~수 TB/s)보다 한 자릿수 이상 느립니다.


<img src="images/figures/new_host_device_transfer.png" width="640">



**전송보다 생성이 싸다**: 큰 데이터를 host에서 만들어 옮기는 것보다 **GPU에서 바로 생성**하는 편이 보통 빠릅니다.

### 조금 더 구체적으로

`make_transfer()`는 (1) NumPy(CPU)로 4096×4096 float32 배열(~64MB)을 생성하고 (2) `cp.asarray`로
PCIe를 통해 전송하는 **두 단계 비용**을 모두 치릅니다. 반면 `make_on_gpu()`는 `cp.random.random`이
GPU 위에서 **cuRAND** 커널을 실행해 device 메모리에 직접 난수를 채우므로 PCIe 전송이 아예 없습니다
— 드는 비용은 커널 launch 오버헤드와 GPU 내부 메모리 대역폭뿐입니다. 이 원리는 난수뿐 아니라
`cp.zeros`/`cp.ones`/`cp.arange`/`cp.empty` 등 **모든 배열 생성 함수**에 동일하게 적용됩니다:
가능하다면 데이터를 만드는 시점부터 GPU에서 만드세요. 실전에서는 파일 I/O나 외부 라이브러리
때문에 host에서 시작할 수밖에 없는 경우가 많은데, 그럴 때조차 **가능한 한 이른 시점에 한 번만
전송**하고 이후 모든 중간 결과는 GPU에 머무르게 하는 것이 핵심 전략입니다.


In [ ]:
N = 4096
# (A) host 생성 후 전송
def make_transfer():
    a = np.random.random((N, N)).astype(np.float32)
    return cp.asarray(a)            # host->device 전송 포함

# (B) GPU에서 직접 생성
def make_on_gpu():
    return cp.random.random((N, N), dtype=cp.float32)

compare('make', make_transfer, make_on_gpu, n_repeat=10, n_warmup=2)
print('=> 같은 데이터라도 GPU 직접 생성이 전송보다 보통 빠릅니다.')

<a id="2"></a>
## 2. 암묵적 전송 주의

CuPy는 다음 상황에서 **조용히 전송·동기화**하여 성능을 갉아먹습니다.
- **출력/표현**: `print(x)`, `str(x)`, f-string 보간
- **파이썬 스칼라/리스트 변환**: `int(x)`, `float(x)`, `bool(x)`, `x.item()`
- **GPU 버전이 없어 NumPy/SciPy로 폴백**하는 함수

또한 `CuPy + NumPy`(rank≥1) 연산은 **에러**입니다(같은 장치에 있어야 함). 0-차원(스칼라) 배열은 일부 예외가 있습니다.

### 조금 더 구체적으로

위 목록에 있는 호출들이 왜 느린지는 **무엇을 하는지 뜯어보면** 분명해집니다. `float(x)`나
`x.item()`은 (1) 그 시점까지 큐에 쌓인 모든 GPU 작업이 끝나길 **동기화 대기**하고, (2) 결과
값 하나를 **`cudaMemcpy`(D2H)** 로 가져오는 두 단계를 거칩니다. `00_intro_env`에서 본 것처럼
GPU 연산은 비동기라 동기화 자체에도 대기 시간이 들고, 전송에는 별도의 고정 지연(latency, 보통
수 마이크로초~수십 마이크로초)이 붙습니다. 루프 안에서 이걸 반복하면(`with_sync`처럼) 그 고정
지연이 **반복 횟수만큼 누적**되어, 정작 GPU가 커널을 실행하는 시간보다 지연 시간의 총합이 더
커지는 역전 현상이 벌어집니다. 아래 실습에서 `n_steps`를 500까지 올려보면 이 누적 효과가 훨씬
극적으로 드러납니다.

`x.sum()`이 CuPy에서 0차원 `cupy.ndarray`를 반환하도록 설계된 이유(`00_intro_env` 3.1절)도 바로
이 문제 때문입니다 — 리덕션 결과를 즉시 스칼라로 바꾸지 않고 GPU에 남겨 두면, `without_sync`처럼
**GPU 위에서 계속 누적**하다가 정말 필요한 마지막 순간에만 한 번 전송할 수 있습니다.

> **왜 `CuPy + NumPy`(rank≥1)가 에러인가**: 두 배열이 서로 다른 메모리 공간에 있으면 CuPy는
> 어느 쪽으로 자동 전송해야 할지 판단할 수 없고, 암묵적으로 아무 방향이나 골라 전송해버리면
> 사용자가 의도치 않은 성능 저하(또는 버그)를 조용히 감수하게 됩니다. CuPy는 이런 "조용한 실수"를
> 막기 위해 **명시적으로 `cp.asarray()`를 호출하도록 강제**합니다. 0차원 스칼라는 브로드캐스트
> 연산에서 흔히 등장하므로 예외적으로 허용됩니다.


<img src="images/figures/new_implicit_transfers.png" width="620">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
n = 1_000_000 # n 을 변경해보세요 e.g.) n = 5_000_000
n_steps = 30 # n_steps 를 변경해보세요 e.g.) n_steps = 500 
x = cp.random.random(n, dtype=cp.float32)

# (나쁨) 루프마다 float()로 스칼라 변환 -> 매번 동기화/전송
def with_sync(x, steps=n_steps):
    acc = 0.0
    for _ in range(steps):
        acc += float((cp.sin(x) + 1).sum())   # float() = 암묵적 전송+동기화
    return acc

# (좋음) GPU에 누적해 두고 마지막에 한 번만 전송
def without_sync(x, steps=n_steps):
    acc = cp.zeros((), dtype=cp.float64)
    for _ in range(steps):
        acc += (cp.sin(x) + 1).sum()
    return float(acc)                          # 마지막 한 번만 전송

compare('sync-in-loop', lambda: with_sync(x), lambda: without_sync(x), n_repeat=5, n_warmup=1)

<a id="3"></a>
## 3. 메모리풀 관찰

CuPy는 `cudaMalloc/Free` 비용을 줄이려 **메모리풀**을 사용합니다(기본 활성). 해제한 배열의 메모리는 풀에 남아 재사용됩니다.
- `cp.get_default_memory_pool()` → `used_bytes()`(사용 중), `total_bytes()`(풀이 보유), `free_all_blocks()`(미사용 블록 반환)

### 조금 더 구체적으로: 왜 캐싱 할당자가 필요한가

`cudaMalloc`/`cudaFree`는 겉보기엔 단순한 메모리 할당이지만, 실제로는 **디바이스 전체를
동기화(implicit synchronization)** 시키는 무거운 드라이버 호출입니다. `sin`, `cos` 같은 원소별
연산 한 번은 마이크로초 단위지만, `cudaMalloc` 한 번은 그보다 한 자릿수 이상 느릴 수 있습니다.
따라서 `for` 루프 안에서 배열을 계속 새로 만들고 버리는 코드는 — 겉으로는 GPU 연산처럼 보여도
— 사실상 매 반복 `cudaMalloc`/`cudaFree` 비용에 발목 잡힙니다.

CuPy의 기본 할당자(`cupy.cuda.MemoryPool`)는 PyTorch의 caching allocator와 같은 전략을 씁니다:
- 배열을 `del`하거나 참조가 사라져도 **실제 `cudaFree`를 호출하지 않고** 그 크기의 블록을 풀에
  보관합니다.
- 다음에 비슷한 크기의 할당 요청이 오면 **재사용**해 `cudaMalloc` 호출 자체를 생략합니다.
- 내부적으로 크기별 **bin**에 블록을 관리하며, 필요하면 큰 블록을 쪼개거나(split) 인접한 빈
  블록을 합쳐(coalesce) 재사용성을 높입니다.

이 설계 때문에 `used_bytes()`(현재 파이썬 배열이 실제로 쓰고 있는 양)와 `total_bytes()`(풀이
OS로부터 받아 보유 중인 양, `nvidia-smi`에 찍히는 값과 유사)가 다릅니다. `del a` 직후에도
`total_bytes()`는 줄지 않는 것이 정상 동작이며, `free_all_blocks()`를 호출해야 미사용 블록이
실제로 `cudaFree`되어 다른 프로세스가 그 메모리를 쓸 수 있게 됩니다.

### 메모리 한도 — `CUPY_GPU_MEMORY_LIMIT`

여러 프로세스나 여러 사용자가 GPU 하나를 공유하는 환경(예: 공용 연구 서버)에서는 CuPy 풀이
필요 이상으로 VRAM을 붙잡고 있으면 다른 프로세스가 OOM(out-of-memory)을 겪을 수 있습니다.
환경변수 **`CUPY_GPU_MEMORY_LIMIT`** 로 풀이 확보할 수 있는 총량을 제한할 수 있습니다.

```bash
# 절대 값(바이트 단위 접미사 허용)
CUPY_GPU_MEMORY_LIMIT="1073741824" python train.py   # 1 GiB
CUPY_GPU_MEMORY_LIMIT="8GB" python train.py

# 전체 VRAM 대비 비율
CUPY_GPU_MEMORY_LIMIT="50%" python train.py
```

한도를 넘는 할당 요청은 풀이 먼저 `free_all_blocks()`에 준하는 정리를 시도한 뒤에도 공간이
부족하면 `OutOfMemoryError`를 던집니다. 코드 안에서도 `mempool.set_limit(size=...)`로 동일하게
설정할 수 있습니다(런타임 중 동적 조정 가능).

### Pinned Memory Pool

지금까지 다룬 것은 **device 메모리풀**이지만, CuPy는 **host 쪽 pinned(고정) 메모리**도
`cp.get_default_pinned_memory_pool()`로 풀링합니다. `cp.asarray`가 내부적으로 pinned 버퍼를
할당해 전송을 가속할 때 이 풀을 사용하며, 동작 원리(캐싱·재사용)는 device 풀과 동일합니다.
pinned memory는 `cudaMemcpyAsync`와 짝을 이뤄야 진가를 발휘하는데, 이 조합은 `06_streams_async`
에서 스트림 오버랩 기법과 함께 다룹니다.

### 커스텀 할당자

기본 풀 대신 다른 할당 전략을 쓰고 싶다면 `cp.cuda.set_allocator(my_allocator)`로 교체할 수
있습니다. 예를 들어 RAPIDS의 **RMM**(RAPIDS Memory Manager)은 서브 할당(sub-allocation)·비동기
풀·다중 GPU 풀 등을 지원해, CuPy와 RAPIDS(cuDF 등)가 **같은 메모리풀을 공유**하도록 연결하는
용도로 흔히 쓰입니다. 커스텀 할당자를 쓰면 여러 라이브러리가 각자 `cudaMalloc`을 따로 호출해
VRAM을 낭비하는 것을 막고, 파편화(fragmentation) 전략을 세밀하게 제어할 수 있습니다.

> **정리**: "메모리풀을 안다"는 것은 결국 `used`(내가 지금 쓰는 양) vs `total`(풀이 쥐고 있는 양)의
> 차이를 이해하고, 필요하면 `free_all_blocks()`/`CUPY_GPU_MEMORY_LIMIT`/커스텀 할당자로 그 차이를
> 제어할 수 있다는 뜻입니다.


> RAPIDS(래피즈)는 엔비디아(NVIDIA)가 개발한 GPU 가속 데이터 과학 및 인공지능(AI) 오픈소스 라이브러리 모음입니다.
> 기존의 파이썬 데이터 분석 도구들(Pandas, Scikit-Learn 등)은 주로 CPU를 사용해서 대용량 데이터를 처리할 때 시간이 오래 걸리지만, RAPIDS는 이 연산들을 GPU로 통째로 옮겨서 수십~수백 배 빠른 속도로 처리할 수 있게 해줍니다.
> RAPIDS의 주요 특징
>  * cuDF (쿠디에프): Pandas와 완전히 똑닮은 문법을 제공하면서, 내부 연산은 GPU로 처리해주는 라이브러리입니다. (데이터프레임 연산 가속)
>  * cuML (쿠엠엘): Scikit-Learn처럼 머신러닝 알고리즘(랜덤 포레스트, K-Means 등)을 GPU에서 빠르게 돌릴 수 있게 해줍니다.
>  * RMM (RAPIDS Memory Manager): GPU 메모리(VRAM)를 효율적으로 쪼개 쓰고(Sub-allocation), CuPy나 cuDF 같은 여러 라이브러리가 메모리 공간을 서로 충돌 없이 안전하게 공유할 수 있도록 관리해 줍니다.


In [ ]:
mempool = cp.get_default_memory_pool()
a = cp.random.random((4096, 4096), dtype=cp.float32)   # ~64MB 할당
print('할당 후  used :', bytes_human(mempool.used_bytes()), '| total:', bytes_human(mempool.total_bytes()))
del a                                                   # 파이썬 참조 해제(풀에는 남음)
print('del 후   used :', bytes_human(mempool.used_bytes()), '| total:', bytes_human(mempool.total_bytes()))
mempool.free_all_blocks()                               # 풀의 미사용 블록 OS 반환
print('free 후  used :', bytes_human(mempool.used_bytes()), '| total:', bytes_human(mempool.total_bytes()))

<a id="4"></a>
## 4. 임시버퍼(temporary) 줄이기 — `out=`

`b = cp.sin(a); c = cp.cos(a); d = b*b + c*c` 처럼 식이 길면 **중간 배열(temporary)** 이 여러 개 생겨 메모리·대역폭을 낭비합니다.
미리 버퍼를 잡고 `out=`에 써 넣으면 할당을 줄일 수 있습니다.

### 조금 더 구체적으로

원소별 연산(elementwise op)은 대부분 **메모리 대역폭 병목(memory-bound)** 입니다 — 연산 자체
(덧셈, sin 등)는 GPU 코어 입장에서 매우 싸지만, 입력을 읽고 출력을 쓰는 **HBM 접근**이 시간을
지배합니다. `slow_step`처럼 매 단계 새 배열을 만들면:
1. `cp.sin(a)` → `a` 읽기 + `b` 쓰기 (새 할당)
2. `cp.cos(a)` → `a` 읽기 + `c` 쓰기 (새 할당)
3. `b*b` → `b` 읽기 + 임시 쓰기, `c*c` → `c` 읽기 + 임시 쓰기, 덧셈 → 두 임시 읽기 + `d` 쓰기
4. `cp.sqrt(d)` → `d` 읽기 + 새 배열 쓰기

각 단계가 **별도의 커널 launch + 별도의 메모리풀 할당 조회 + 별도의 HBM read/write 왕복**을
일으킵니다. `out=`을 지정하면 (a) 새 할당이 사라져 메모리풀 조회 비용이 줄고, (b) 이미 존재하는
버퍼에 덮어써 캐시 지역성이 좋아지며, (c) 무엇보다 **불필요한 배열 생성 자체가 없어져** 풀이
관리해야 할 블록 수가 줄어듭니다. 20,000,000개 float32 배열이면 원소 하나당 4바이트이므로 temporary
하나마다 약 80MB의 추가 쓰기가 발생한다는 점을 상상하면 왜 `out=`이 체감될 만큼 빨라지는지 감이
옵니다.

> 더 근본적인 해결책은 여러 원소별 연산을 **커널 하나로 융합(kernel fusion)**해 중간 결과를
> HBM에 아예 쓰지 않는 것입니다. CuPy는 `cp.fuse` 데코레이터나 `ElementwiseKernel`로 이를
> 지원하며, 사용자 정의 커널 작성은 Day 2(단원 5~6)에서 다룹니다. `out=`은 코드를 거의 바꾸지
> 않고 얻을 수 있는 **가장 저렴한 최적화**이고, 융합은 한 단계 더 나아간 선택지라고 생각하면 됩니다.


In [ ]:
a = cp.random.random(1_000, dtype=cp.float32)

def slow_step(a):
    b = cp.sin(a); c = cp.cos(a); d = b*b + c*c
    return cp.sqrt(d)

def fast_step(a, b, c, d):
    cp.sin(a, out=b); cp.cos(a, out=c)
    cp.multiply(b, b, out=d); d += c*c
    cp.sqrt(d, out=d); return d

b = cp.empty_like(a); c = cp.empty_like(a); d = cp.empty_like(a)
r_slow = bench(lambda: slow_step(a), n_repeat=10, name='temporary 많음')
r_fast = bench(lambda: fast_step(a, b, c, d), n_repeat=10, name='out= 재사용')
print_bench(r_slow); print_bench(r_fast)
print(f'speedup: {gpu_ms(r_slow)/gpu_ms(r_fast):.2f}x')

<a id="5"></a>
## 5. 전송 병목 비교

루프 안에서 반복적으로 host로 가져오면(`asnumpy`/`float`) 전송이 누적되어 병목이 됩니다.
2절과 같은 원리지만, 여기서는 더 큰 데이터로 전송 비용의 크기를 봅니다.

### 조금 더 구체적으로

2절의 `with_sync`/`without_sync` 실험이 "동기화·전송이 왜 나쁜가"를 원리 차원에서 보여줬다면,
여기서는 데이터 크기를 10배(1M → 10M 원소)로 키워 **절대적인 전송 비용의 규모**를 체감합니다.
`heavy`는 매 스텝 `float(...)`로 스칼라 하나만 가져오는 것처럼 보이지만, 그 전에 `(cp.sin(x)+1).sum()`
전체가 완료될 때까지 **동기화**를 기다려야 하므로 GPU 파이프라인이 스텝마다 끊깁니다. `light`는
전부 GPU에 누적해 두었다가 **딱 한 번**만 host로 내보내므로, GPU가 스텝 사이에 쉬지 않고 계속
작업을 큐에 채워 넣을 수 있습니다(파이프라이닝). `00_intro_env` 7절에서 측정한 "작은 전송은
고정 지연이 지배하고, 큰 전송은 대역폭이 지배한다"는 원리를 떠올리면, 여기서 문제가 되는 것은
전송 데이터 자체의 크기가 아니라 **전송 "횟수"** 라는 점이 명확해집니다. 이 반복 동기화를 줄이는
또 다른 방법(비동기 전송 + 스트림 오버랩)은 `06_streams_async`에서 배웁니다.


In [ ]:
x = cp.random.random(10_000_000, dtype=cp.float32)

def heavy(x, steps=20):   # 매 스텝 host 전송
    acc = 0.0
    for _ in range(steps):
        acc += float((cp.sin(x) + 1).sum())
    return acc

def light(x, steps=20):   # GPU 누적, 마지막에 1회 전송
    acc = cp.zeros((), dtype=cp.float64)
    for _ in range(steps):
        acc += (cp.sin(x) + 1).sum()
    return float(acc)

compare('transfer', lambda: heavy(x), lambda: light(x), n_repeat=3, n_warmup=1)

<a id="6"></a>
## 6. 프로파일링 도구

성능 개선은 **측정 → 가설 → 수정** 순서로 합니다. CuPy가 제공하는 도구:
- **`cupyx.profiler.benchmark`** (= `course_utils.bench`): 워밍업·동기화·반복평균 자동.
- **`cupyx.profiler.time_range`**: 코드 구간을 **NVTX 범위**로 표시(데코레이터/컨텍스트 매니저). Nsight Systems 타임라인에 나타남.
- **`cupyx.profiler.profile`**: 프로파일러 캡처 구간을 켜고 끄는 컨텍스트 매니저. `nsys --capture-range=cudaProfilerApi` 와 함께 사용.

Jupyter notebook console에서 다음을 수행합니다.

### 조금 더 구체적으로: 세 도구는 서로 다른 질문에 답한다

- **`benchmark`**: "이 함수가 **평균적으로 몇 ms** 걸리는가?"에 답합니다. `00_intro_env`에서
  이미 봤듯 CUDA 이벤트로 GPU 완료 시점을 정확히 재고, 워밍업으로 일회성 오버헤드(컨텍스트 초기화,
  커널 JIT 컴파일)를 배제합니다. **숫자 하나**로 요약되는 리포트가 필요할 때 씁니다.
- **`time_range`**: "이 구간이 전체 타임라인에서 **어디에, 얼마나** 걸리는가?"에 답합니다. 코드에
  라벨(이름 + 색상)을 붙여 **NVTX**(NVIDIA Tools Extension) 마커를 커널 실행과 함께 기록하고,
  이를 Nsight Systems(`nsys`)로 열면 CPU 실행·커널 실행·메모리 전송이 **같은 시간축 위에서**
  겹쳐 보입니다. 숫자 하나로는 안 보이는 "커널 사이 빈 틈(GPU가 노는 시간)"이나 "전송과 연산이
  겹치는지"를 눈으로 확인할 때 씁니다.
- **`profile`**: `nsys profile` 자체가 프로세스 전체를 처음부터 끝까지 기록하면 로그가 방대해지고
  노이즈(import, 초기화 등)가 섞입니다. `profile()` 컨텍스트는 `cudaProfilerStart/Stop`을 호출해
  **"진짜 측정하고 싶은 구간만"** 캡처하도록 nsys에 신호를 보냅니다 — `nsys profile
  --capture-range=cudaProfilerApi`와 짝을 이뤄야 동작합니다.

세 도구는 배타적이지 않고 함께 씁니다: 먼저 `benchmark`로 "느리다"는 신호를 잡고, `time_range`로
어느 구간이 문제인지 라벨링한 뒤, `profile`로 그 구간만 nsys에 캡처해 Nsight Systems GUI에서
자세히 들여다보는 흐름이 실전 워크플로입니다. 아래 셀들은 이 워크플로를 실제로 실행하기 위해
**Nsight Systems(`nsys`)를 설치**하고, `time_range`로 라벨링한 예제 스크립트를 nsys로 프로파일링해
`.nsys-rep` 파일을 만드는 과정을 안내합니다.


1. 최신 Ubuntu 24.04용 cuda-keyring 패키지 다운로드

In [ ]:
! wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-keyring_1.1-1_all.deb -O cuda-keyring.deb 


--2026-08-12 06:42:09--  https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-keyring_1.1-1_all.deb
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.44.175.114, 23.44.175.100, 23.44.175.105, ...
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.44.175.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4328 (4.2K) [application/x-deb]
Saving to: ‘cuda-keyring.deb’

cuda-keyring.deb    100%[===================>]   4.23K  --.-KB/s    in 0s      

2026-08-12 06:42:09 (1.21 GB/s) - ‘cuda-keyring.deb’ saved [4328/4328]

--2026-08-12 06:42:10--  https://developer.nvidia.com/downloads/assets/tools/secure/nsight-systems/2026_4/NsightSystems-linux-cli-public-2026.4.1.191-3860507.deb
Resolving developer.nvidia.com (developer.nvidia.com)... 23.59.88.228, 23.59.88.238, 23.59.88.236, ...
Connecting to developer.nvidia.com (developer.nvidia.com)|23.59.88.228|:443... connected.
HTTP request s

2. dpkg 명령어로 keyring 설치 (NVIDIA 저장소 인증 키 등록)

In [ ]:
! dpkg -i cuda-keyring.deb


(Reading database ... 24745 files and directories currently installed.)
Preparing to unpack cuda-keyring.deb ...
Unpacking cuda-keyring (1.1-1) over (1.1-1) ...
Setting up cuda-keyring (1.1-1) ...
Selecting previously unselected package nsight-systems-cli-2026.4.1.
(Reading database ... 24745 files and directories currently installed.)
Preparing to unpack nsight-systems.deb ...
Unpacking nsight-systems-cli-2026.4.1 (2026.4.1.191-264138605071v0) ...
Setting up nsight-systems-cli-2026.4.1 (2026.4.1.191-264138605071v0) ...
update-alternatives: using /opt/nvidia/nsight-systems-cli/2026.4.1/target-linux-x64/nsys to provide /usr/local/bin/nsys (nsys) in auto mode


3. 사용한 임시 deb 파일 삭제

In [ ]:
! rm cuda-keyring.deb


4. NVIDIA 신규 저장소가 반영되도록 apt 업데이트

In [13]:
! rm -f /etc/apt/sources.list.d/cuda*.list
! rm -f /etc/apt/sources.list.d/nvidia*.list
! apt-get clean
! apt-get update

Hit:1 https://packagecloud.io/github/git-lfs/ubuntu noble InRelease            
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease                         
Hit:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done


5. CUDA 버전별 nsight-systems 패키지 설치

In [15]:
! apt-get install nsight-systems

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package nsight-systems is not available, but is referred to by another package.
This may mean that the package is missing, has been obsoleted, or
is only available from another source

E: Package 'nsight-systems' has no installation candidate


In [14]:
# 에러 나면 apt-get install -y nsight-systems 통해서 세부 버전 확인
! apt-get install -y nsight-systems-2026.4.1

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package nsight-systems-2026.4.1
E: Couldn't find any package by glob 'nsight-systems-2026.4.1'
E: Couldn't find any package by regex 'nsight-systems-2026.4.1'


In [ ]:
# 에러 나면 Nsights System 만 설치

! wget https://developer.nvidia.com/downloads/assets/tools/secure/nsight-systems/2026_4/NsightSystems-linux-cli-public-2026.4.1.191-3860507.deb -O nsight-systems.deb 
! dpkg -i nsight-systems.deb
! rm nsight-systems.deb

6. nsys 설치 확인

In [16]:
! nsys --version

NVIDIA Nsight Systems version 2026.4.1.191-264138605071v0


7. 4에서 저장소 에러나올 경우 아래 커맨드로 [signed-by=...] 이 없는 중복 파일 확인 후 삭제

In [ ]:
! grep -RIn "developer.download.nvidia.com/compute/cuda" /etc/apt/sources.list /etc/apt/sources.list.d/ 2>/dev/null

`!nsys profile python ./cupy/05_6_profiling_test.py` 실행후, 출력파일(확장자 nsys-rep) 다운로드 하신뒤, 로컬 컴퓨터에서 nsight 프로그램을 실행해서 타임라인을 확인해보세요. `time_range`로 구간을 라벨링하세요.

In [17]:
%%writefile 05_6_profiling_test.py
import cupy as cp
import time
from cupyx.profiler import time_range
from cupy.cuda import profiler

print("--- NVTX 프로파일링 테스트 시작 ---")

# 1.  데이터 준비
x = cp.random.random(20_000_000, dtype=cp.float32)

# 2. nsys 프로파일러 수집 시작 명령 
profiler.start()

# [테스트 1] 컨텍스트 매니저 방식
print("Step 1: Context Manager 테스트 중...")
with time_range('MY_CONTEXT_RANGE', color_id=1):
    for _ in range(20):
        y = (cp.sin(x) + 1).sum()
    cp.cuda.Device().synchronize() # NVTX 영역이 닫히기 전 GPU 연산 보장

time.sleep(0.5) # 타임라인 구분용 공백 시간

# [테스트 2] 데코레이터 방식
@time_range('MY_DECORATOR_RANGE', color_id=3)
def run_stage(data):
    for _ in range(20):
        data = (cp.cos(data) ** 2).sum()
    return data.sum()

print("Step 2: Decorator 테스트 중...")
res = run_stage(x)
cp.cuda.Device().synchronize()

# 3. 프로파일러 수집 종료 명령
profiler.stop()

print("--- 테스트 완료! 결과를 nsys-rep 파일로 저장합니다. ---")


Writing 05_6_profiling_test.py


In [18]:
! nsys profile --force-overwrite true -o profiling_test python 05_6_profiling_test.py

--- NVTX 프로파일링 테스트 시작 ---
Step 1: Context Manager 테스트 중...
Step 2: Decorator 테스트 중...
--- 테스트 완료! 결과를 nsys-rep 파일로 저장합니다. ---
Generating '/tmp/nsys-root/nsys-report-aec3.qdstrm'
[1/1] [========================100%] profiling_test.nsys-rep
Generated:
	/root/cupy/profiling_test.nsys-rep


In [20]:
! ls -lh profiling_test.nsys-rep

-rw-rw-r--. 1 root root 700K Aug 12 06:44 profiling_test.nsys-rep


### 6.1 CUB/cuTENSOR 백엔드 — "공짜" 가속

리덕션(`sum`, `prod`, `min/max`, `argmin/argmax`, `cumsum` 등)은 **CUB**/**cuTENSOR** 백엔드로 더 빨라질 수 있습니다.
환경변수 **`CUPY_ACCELERATORS`** 로 선택합니다(시도 순서대로). 예: 세션을 `CUPY_ACCELERATORS=cub,cutensor python` 으로 시작.

- CuPy **v11+ 는 기본으로 CUB를 사용**합니다(끄려면 `CUPY_ACCELERATORS=""`).
- cuTENSOR는 별도 설치 시 이항 ufunc·리덕션·텐서 축약을 가속합니다.
- 환경변수는 **프로세스 시작 전에** 설정해야 하며, 데이터 레이아웃(연속 축 여부)에 따라 효과가 달라 — 항상 벤치마크로 확인하세요.

### 조금 더 구체적으로

CuPy의 기본 리덕션 커널은 범용적으로 동작하도록 작성된 반면, **CUB**(CUDA UnBound)는 NVIDIA가
직접 만든 **템플릿 기반 프리미티브 라이브러리**로, warp/block 단위 리덕션·정렬·스캔 등을 하드웨어
세대별로 세밀하게 튜닝해 제공합니다. 같은 `sum()`이라도 CUB 경로를 타면 block 단위로 먼저 부분합을
계산하고(segmented reduction), 그 결과를 다시 합치는 **2단계 트리 리덕션**을 아키텍처에 맞춰
최적화된 방식으로 수행해 범용 커널보다 빠른 경우가 많습니다. cuTENSOR는 여기서 한 걸음 더 나아가
텐서 축약(contraction)·이항 연산까지 아키텍처별 튜닝 커널로 대체합니다. `CUPY_ACCELERATORS`에
넣을 수 있는 값은 이 둘 외에도 `cutensornet`, `cusparselt` 등이 있으며, 쉼표로 나열한 **순서대로
시도**해 사용 가능한 첫 백엔드를 채택합니다.

> **왜 별도 스크립트(`cubtest.py`)로 실행하는가**: `CUPY_ACCELERATORS`는 CuPy가 **CUDA 컨텍스트를
> 초기화하는 시점 이전**에 읽히는 환경변수입니다. 이 노트북 커널은 이미 `import cupy`를 마친
> 상태이므로, 지금 `os.environ`을 바꿔도 이미 뜬 프로세스에는 반영되지 않습니다. 그래서
> `%%writefile`로 별도 파일을 만들고 `!python cubtest.py`처럼 **새 프로세스**로 실행해야 환경변수
> 변경 효과를 관찰할 수 있습니다 — 이는 CUDA 관련 환경변수 대부분(`CUPY_GPU_MEMORY_LIMIT`,
> `CUPY_CACHE_DIR` 등)에 공통되는 규칙입니다.


In [ ]:
%%writefile cubtest.py

import os
#os.environ["CUPY_ACCELERATORS"] = ""  
os.environ["CUPY_ACCELERATORS"] = "cub"  
import cupy as cp
from course_utils import bench, print_bench

# 현재 백엔드로 큰 배열 리덕션 측정 (CUPY_ACCELERATORS 설정에 따라 속도가 달라짐)
a = cp.random.random((256, 256, 256), dtype=cp.float32)
print_bench(bench(a.sum, n_repeat=50, n_warmup=5, name='sum (현재 백엔드)'))
# 비교 실험: 위의 environ 주석을 번갈아가면서 실행하고, '!python ./cupy/cubtest.py'를 각각 수행해보세요.  

<a id="7"></a>
## 7. 연습문제 — 비효율 파이프라인 2배 개선

아래 `pipeline_slow`는 (1) 데이터를 host에서 만들어 전송하고, (2) temporary가 많으며, (3) 루프마다 스칼라를 host로 가져옵니다.
세 가지를 고쳐 `pipeline_fast`를 만들고, `compare`로 **2배 이상** 빨라지는지 확인하세요.

이 세 가지 문제는 각각 1절(GPU 생성 vs 전송), 4절(`out=`), 2·5절(암묵적 전송·동기화)에서
개별적으로 다뤘던 안티패턴을 **한 함수 안에 모두 모아놓은 것**입니다. 즉 이 연습문제는 새로운
개념이 아니라, 지금까지 배운 세 가지 원칙을 동시에 적용해보는 종합 실습입니다. 하나씩 고쳐가며
`compare`로 중간 결과를 찍어보면 어떤 수정이 성능에 가장 크게 기여하는지 감을 잡을 수 있습니다.


In [ ]:
def pipeline_slow(N=10_000_000, steps=15):
    a = cp.asarray(np.random.random(N).astype(np.float32))   # (1) host 생성+전송
    total = 0.0
    for _ in range(steps):
        b = cp.sin(a); c = cp.cos(a); d = b*b + c*c           # (2) temporary 다수
        total += float(cp.sqrt(d).sum())                      # (3) 매 스텝 host 전송
    return total

def pipeline_fast(N=10_000_000, steps=15):
    # TODO: (1) cp.random로 GPU 직접 생성  (2) out= 버퍼 재사용  (3) GPU 누적 후 1회 전송
    raise NotImplementedError

# compare('pipeline', lambda: pipeline_slow(), lambda: pipeline_fast(), n_repeat=3, n_warmup=1)

<details>
<summary>💡 해답 보기</summary>

```python
def pipeline_fast(N=10_000_000, steps=15):
    a = cp.random.random(N, dtype=cp.float32)        # (1) GPU 직접 생성
    b = cp.empty_like(a); c = cp.empty_like(a); d = cp.empty_like(a)
    total = cp.zeros((), dtype=cp.float64)            # (3) GPU 누적
    for _ in range(steps):
        cp.sin(a, out=b); cp.cos(a, out=c)            # (2) out= 재사용
        cp.multiply(b, b, out=d); d += c*c
        cp.sqrt(d, out=d)
        total += d.sum()
    return float(total)                               # 마지막 1회만 전송

compare('pipeline', lambda: pipeline_slow(), lambda: pipeline_fast(), n_repeat=3, n_warmup=1)
```

세 가지 개선(생성·temporary·전송)이 합쳐져 보통 2배 이상 빨라집니다. 어떤 요인이 가장 컸는지 하나씩 켜고 꺼 보세요. 경험적으로는 (3) 루프 내 host 전송 제거가 가장 큰 단일 효과를 내는 경우가 많은데, 이는 매 스텝 동기화 대기가 GPU 파이프라인을 통째로 끊기 때문입니다. (1) GPU 직접 생성과 (2) `out=`은 그보다는 작지만 꾸준한 개선을 더합니다 — 세 요인을 개별적으로 껐다 켜보면 이 상대적 기여도를 직접 확인할 수 있습니다.
</details>

<a id="8"></a>
## 8. (실전) Power Iteration — 메모리 공간 적용

지금까지 배운 원칙(전송 최소화·GPU에서 생성·암묵적 동기화 회피)을 **현실적 알고리즘**에 적용합니다.
**거듭제곱 반복법(Power Iteration)** 은 행렬의 **최대 고유값**을 구합니다: `y = A x` → 정규화를 반복.
포팅은 `np.`→`cp.` 치환이며, 핵심은 **A를 GPU에서 직접 생성**하고 반복 중 host 전송을 피하는 것입니다.
(GTC Memory Spaces 예제를 본 과정 형식으로 재구성)

### 알고리즘 배경

Power Iteration은 `x_{k+1} = A x_k / ||A x_k||`를 반복하면 `x_k`가 **가장 큰 절대값을 갖는
고유값(dominant eigenvalue)에 대응하는 고유벡터**로 수렴한다는 선형대수 정리를 이용합니다.
수렴 속도는 1등과 2등 고유값의 비율 `|λ2/λ1|`에 좌우됩니다 — 이 비율이 작을수록(두 고유값이
멀리 떨어질수록) 빠르게 수렴하고, 1에 가까울수록(두 고유값이 비슷할수록) 반복이 오래 걸립니다.
여기서는 대칭 랜덤 행렬(`make_spd`)을 써서 300회 반복이면 충분히 수렴하도록 구성했고, 결과를
`np.linalg.eigvalsh`(대칭행렬 전용 고유값 계산)의 최대값과 대조해 검증합니다.

### 이 예제가 "메모리 공간" 단원의 마무리인 이유

Power Iteration은 매 반복마다 `A @ x`(행렬-벡터곱, `n×n` 행렬이면 GPU 코어에게는 가벼운 연산)와
정규화(`norm`)만 수행하고, **300번의 반복 동안 host로 단 한 번도 나가지 않습니다**(마지막
레일리 몫(Rayleigh quotient) 계산에서 딱 한 번 `float()`로 전송). 만약 순진하게 포팅해 매
반복마다 `float(...)`로 중간 고유값 추정치를 출력하거나 `x`를 host로 가져와 검사했다면, 2·5절에서
본 것처럼 동기화 지연이 300번 누적되어 GPU 가속 효과가 거의 사라졌을 것입니다. 즉 이 예제는
"어떻게 빠른 행렬곱을 짜는가"가 아니라 — 행렬곱 자체는 이미 cuBLAS가 최적화해 줍니다 —
**"메모리 공간 규칙(생성 위치·전송 시점)을 지키면 최적화된 커널의 성능이 그대로 살아난다"**는
이 노트북 전체의 메시지를 반복 알고리즘으로 확인하는 것이 핵심입니다.


In [ ]:
def make_spd(xp, n):
    M = xp.random.random((n, n)).astype(xp.float32)
    return (M + M.T) / 2          # 대칭 -> 실수 고유값 보장

def power_iteration(xp, A, iters=300):
    x = xp.ones(A.shape[0], dtype=A.dtype)
    for _ in range(iters):
        y = A @ x
        x = y / xp.linalg.norm(y)   # 반복 중 host 전송 없음 (모두 GPU에 머무름)
    return float((x @ (A @ x)) / (x @ x))   # 마지막에 한 번만 스칼라 전송

n = 2000

# --- 실행 및 검증 ---
A_np = make_spd(np, n)
# [수정 사항] CPU 데이터를 전송하는 대신, GPU에서 직접 무작위 대칭 행렬을 바로 생성합니다.
A_cp = make_spd(cp, n)

lam_np = power_iteration(np, A_np)
lam_cp = power_iteration(cp, A_cp)
ref = float(np.linalg.eigvalsh(A_np).max())
print(f'CPU λ={lam_np:.5f} | GPU λ={lam_cp:.5f} | ref(eigvalsh)={ref:.5f}')


In [ ]:
# CPU vs GPU 속도 (같은 행렬 A_np 입력으로 공정 비교)
compare('power_iter', 
        lambda: power_iteration(np, A_np),
        lambda: power_iteration(cp, A_cp), n_repeat=3, n_warmup=1)
print('=> A를 GPU에서 직접 생성하면 host->device 전송까지 아낍니다 (make_spd(cp, n)).')

## 🧪 추가 연습 & 실험

**연습 — `out=`로 temporary 제거**: 아래 식을 `out=` 버퍼로 다시 써서 임시배열을 없애고 속도를 비교하세요.
식: `r = sqrt(exp(-a) + log1p(a))`

4절에서 연습한 것과 같은 패턴입니다 — 식에 등장하는 중간 결과(`exp(-a)`, `log1p(a)`, 합, 제곱근)
개수만큼 temporary가 생기므로, 미리 잡아둔 두 개의 버퍼(`t1`, `t2`)를 돌려가며 `out=`에 덮어써
새 할당을 완전히 없애보세요.


In [ ]:
a = cp.random.random(1_000, dtype=cp.float32)
def f_slow(a):
    return cp.sqrt(cp.exp(-a) + cp.log1p(a))
def f_fast(a, t1, t2):
    # TODO: cp.exp(-a,out=t1); cp.log1p(a,out=t2); t1+=t2; cp.sqrt(t1,out=t1); return t1
    raise NotImplementedError

t1 = cp.empty_like(a); t2 = cp.empty_like(a)
# r_slow=bench(lambda:f_slow(a),name='slow'); r_fast=bench(lambda:f_fast(a,t1,t2),name='fast')
# print_bench(r_slow); print_bench(r_fast); print('speedup', round(gpu_ms(r_slow)/gpu_ms(r_fast),2))

<details><summary>💡 해답 보기</summary>

```python
def f_fast(a, t1, t2):
    cp.exp(cp.negative(a), out=t1)   # 또는 cp.exp(-a, out=t1)
    cp.log1p(a, out=t2)
    t1 += t2
    cp.sqrt(t1, out=t1)
    return t1

t1 = cp.empty_like(a); t2 = cp.empty_like(a)
r_slow = bench(lambda: f_slow(a), name='slow')
r_fast = bench(lambda: f_fast(a, t1, t2), name='fast')
print_bench(r_slow); print_bench(r_fast)
print('speedup', round(gpu_ms(r_slow)/gpu_ms(r_fast), 2))
```

두 버퍼만으로 네 번의 연산(exp, log1p, 덧셈, sqrt)을 모두 제자리에서 처리했습니다 — `t1 += t2`도 새 배열을 만들지 않는 in-place 연산이라는 점에 주목하세요.
</details>

**실험 — 메모리풀 증가 관찰**: 크기를 키우며 `used/total`을 출력하고, `free_all_blocks()` 전후를 비교하세요.

3절에서 설명한 caching allocator의 동작을 직접 눈으로 확인하는 실험입니다. 예측해보세요: 크기가
1M→64M(64배)으로 커지는 동안 `used_bytes()`는 매번 새 배열 크기만큼만 늘고 이전 배열의 공간은
`del` 시점에 풀로 반환되어 `total_bytes()`에 그대로 누적되지 않을 수 있습니다(이전 블록이 재사용
가능한 크기라면). `free_all_blocks()` 이후에는 `used`와 `total`이 모두 0에 가까워지는지 — 즉
풀이 실제로 OS에 메모리를 반환했는지 — 확인하세요.


In [ ]:
mp = cp.get_default_memory_pool()
for n in [1, 4, 16, 64]:
    x = cp.random.random(n*1_000_000, dtype=cp.float32)
    print(f'{n:>3}M elems | used {bytes_human(mp.used_bytes()):>10} | total {bytes_human(mp.total_bytes()):>10}')
    del x
mp.free_all_blocks()
print('free 후   | used', bytes_human(mp.used_bytes()), '| total', bytes_human(mp.total_bytes()))

<a id="8"></a>
## 9. 체크포인트

- [ ] host/device 전송과 'GPU에서 생성' 이점을 안다
- [ ] 암묵적 전송(print/float/.item/폴백)을 식별하고 피한다
- [ ] 메모리풀의 used/total/free_all_blocks를 관찰했다
- [ ] `CUPY_GPU_MEMORY_LIMIT`과 커스텀 할당자로 메모리 사용을 제어하는 방법을 안다
- [ ] `out=`으로 temporary를 줄여 속도를 높였다
- [ ] `benchmark`/`time_range`/`profile`의 용도를 구분한다
- [ ] `CUPY_ACCELERATORS`(CUB)로 리덕션을 가속하는 법을 안다
- [ ] 연습: 파이프라인을 2배 이상 개선했다

다음: **`06_streams_async`** — 스트림·이벤트·비동기 전송으로 연산과 전송을 겹칩니다.
